# 20 - Les points d'extension de reecriture du DSL

**Navigation** : [Index SymbolicAI](../../README.md) | [Serie Z3](README.md)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Expliquer ce qu'est un **point d'extension de reecriture** dans un binding de solveur, et a quel moment il s'applique ;
2. Ecrire un **reecrivain global** qui ajoute une contrainte a un theoreme, et le voir changer le verdict ;
3. Ecrire un **reecrivain de predicat** qui traduit une methode inconnue du DSL en une contrainte existante ;
4. Reproduire les **gardes-fous du contrat** qui protegent ces deux mecanismes.

### Pre-requis

- .NET Interactive (noyau `.net-csharp`) ; la version reellement utilisee est ecrite par la premiere cellule de code
- Les carnets `02` (encodage Sudoku), `17` (coeurs UNSAT) et `19` (choix du solveur)

### Duree estimee : 45-55 minutes

***

> **Stack : A** - ce carnet reste dans le DSL (`Z3.Linq`). Ce qui change, c'est le moment ou la formule est construite : jusqu'ici le binding traduisait un theoreme que vous ecrivez ; ici, il vous laisse **ecrire vos propres regles de traduction**.

Le binding `Z3.Linq` expose **deux points d'extension** de reecriture, declares par des attributs :

| Attribut | Port sur | Effet |
|---|---|---|
| `[TheoremGlobalRewriter(typeof(R))]` | le **type d'environnement** | `R` recoit toutes les contraintes du theoreme et renvoie la liste a faire assertor |
| `[TheoremPredicateRewriter(typeof(R))]` | une **methode de predicat** | `R` recoit l'appel de methode et renvoie l'appel a traduire a sa place |

Mesure de l'absence, sur `origin/main` avant ce carnet :

```
git grep -o -E "ITheoremGlobalRewriter|ITheoremPredicateRewriter|TheoremGlobalRewriter|TheoremPredicateRewriter" \
    origin/main -- 'MyIA.AI.Notebooks/**/*.ipynb' | wc -l
0
```

Le fork les implemente pourtant (`solutions/Z3.Linq/ITheoremGlobalRewriter.cs`, `ITheoremPredicateRewriter.cs`, `TheoremGlobalRewriterAttribute.cs`, `TheoremPredicateRewriterAttribute.cs`, et les points d'appel `Theorem.cs:396-418` et `ExpressionVisitor.cs:446-471`) : c'est une capacite **publique, reelle et jamais exposee**.

In [1]:
#r "../Z3.Linq/.deploy/Microsoft.Z3.dll"
#r "../Z3.Linq/.deploy/ExpressionUtils.dll"
#r "../Z3.Linq/.deploy/Z3.Linq.dll"
using Z3.Linq; using Microsoft.Z3; using System; using System.Collections.Generic; using System.Linq; using System.Runtime.InteropServices;

// --- Environnement ---
// Ces lignes alimentent les sorties commitees : un resultat sans environnement
// decrit n'est pas une preuve, c'est une anecdote.
Console.WriteLine($"Framework      : {RuntimeInformation.FrameworkDescription}");
Console.WriteLine($"Architecture   : {RuntimeInformation.ProcessArchitecture}");
Console.WriteLine($"Assembly Z3    : {typeof(Context).Assembly.GetName().Version}");

// --- L'instance : la meme grille que le carnet 19 (30 indices sur 81 cases) ---
const string Puzzle = "530070000600195000098000060800060003400803001700020006060000280000419005000080079";
Console.WriteLine($"Indices de la grille : {Puzzle.Count(c => c != '0')} / 81 cases");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Framework      : .NET 10.0.13


Architecture   : X64


Assembly Z3    : 4.12.2.0


Indices de la grille : 30 / 81 cases


## 1. Le reecrivain global : ecrire une regle qui s'applique a tout le theoreme

Un reecrivain global est une classe qui implemente `ITheoremGlobalRewriter`. Le binding l'instancie, lui passe **toutes** les contraintes du theoreme, et fait assertor ce qu'elle renvoie.

Deux consequences importantes, et toutes deux mesurees dans ce carnet :

- le reecrivain travaille sur l'**arbre d'expressions**, avant toute traduction vers Z3 : il construit lui-meme la contrainte qu'il ajoute ;
- l'attribut est lu sur le **type d'environnement** du theoreme (`typeof(T)`). Un type d'environnement par configuration.

La regle que nous allons ecrire est simple et verifiable : **interdire la valeur 5 dans la case 0**. La case 0 de notre grille vaut exactement 5 (c'est un indice de la grille) : la regle contredit donc un indice, et le theoreme doit devenir insoluble.

Pour que l'index de la contrainte soit lisible, on l'affiche telle que le reecrivain l'a construite.

In [2]:
using System.Linq.Expressions;

// --- Les parametres de la regle ---
// Le binding instancie le reecrivain SANS argument : une regle parametrique se
// configure donc par un etat statique, lu au moment de la reecriture.
public static class Regle
{
    public static int CaseInterdite = 0;
    public static int ValeurInterdite = 5;
}

// --- Le reecrivain global ---
// Il recoit toutes les contraintes, en ajoute une, et renvoie la nouvelle liste.
// La contrainte ajoutee est construite sur l'arbre d'expressions : on lit la
// propriete `Cells` du type d'environnement et on l'indexe.
public class ForbidValueRewriter : ITheoremGlobalRewriter
{
    public IEnumerable<LambdaExpression> Rewrite(IEnumerable<LambdaExpression> constraints)
    {
        var liste = constraints.ToList();
        var modele = liste[0];
        var t = modele.Parameters[0];
        var cases = Expression.Property(t, "Cells");
        var getItem = typeof(List<int>).GetMethod("get_Item")!;
        var caseInterdite = Expression.Call(cases, getItem, Expression.Constant(Regle.CaseInterdite));
        var corps = Expression.NotEqual(caseInterdite, Expression.Constant(Regle.ValeurInterdite));
        var ajoutee = Expression.Lambda(modele.Type, corps, t);
        Console.WriteLine($"  [ForbidValueRewriter] {liste.Count} contraintes recues ; ajout : {ajoutee.Body}");
        liste.Add(ajoutee);
        return liste;
    }
}


warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



In [3]:
using System.Linq;

// --- Les types d'environnement ---
// Le binding lit l'attribut de reecriture sur le TYPE d'environnement : il faut
// donc un type par configuration. La propriete `Cells` doit etre declaree sur
// le type lui-meme (constate par une sonde hors carnet : heritee d'une classe de base, le binding echoue
// sur « Unknown parameter encountered: Cells. »).
public class SudokuArray
{
    public List<int> Cells { get; set; } = Enumerable.Repeat(0, 81).ToList();
}

[TheoremGlobalRewriter(typeof(ForbidValueRewriter))]
public class SudokuArrayAvecRegle
{
    public List<int> Cells { get; set; } = Enumerable.Repeat(0, 81).ToList();
}

### Les types d'environnement, et pourquoi il y en a plusieurs

Le binding lit l'attribut de reecriture sur le **type** de l'environnement : `typeof(T)` du theoreme. Un type par configuration, donc : `SudokuArray` pour l'enonce nu, `SudokuArrayAvecRegle` pour l'enonce avec regle, `SudokuArrayGarde` pour le cas fautif de la section 3.

Chaque type declare **sa propre** propriete `Cells`. Une propriete heritee d'une classe de base echoue sur « Unknown parameter encountered: Cells. » (message constate par une sonde hors carnet) : la contrainte de conception du binding se paie des deux cotes - le type porte l'attribut, et il porte sa propriete.

In [4]:
using System.Linq;

// --- Un type d'environnement dont l'attribut pointe sur un type qui n'est pas
//     un reecrivain : c'est la garde-fou de la section 3, on la prepare ici. ---
public class NotAGlobalRewriter { }

[TheoremGlobalRewriter(typeof(NotAGlobalRewriter))]
public class SudokuArrayGarde
{
    public List<int> Cells { get; set; } = Enumerable.Repeat(0, 81).ToList();
}

In [5]:
using System.Linq;

// --- L'encodage du carnet 02, repris tel quel ---
// Deux methodes presque identiques : le binding resout les membres de
// l'environnement sur le type CONCRET de l'arbre d'expressions. Un encodeur
// generique ne fonctionne pas (constate par une sonde hors carnet : « Unknown parameter encountered: Cells. »)
// : il faut donc un encodeur par type d'environnement.
public static class Sudoku09
{
    // 81 contraintes de domaine, 9 lignes, 9 colonnes, 9 boites, puis les indices.
    public static Theorem<SudokuArray> Charger(Z3Context ctx, string puzzle)
    {
        var th = ctx.NewTheorem<SudokuArray>();
        var indices = Enumerable.Range(0, 9).ToArray();

        for (int i = 0; i < 9; i++)
            for (int j = 0; j < 9; j++)
            {
                var i1 = i; var j1 = j;
                th = th.Where(s => s.Cells[i1 * 9 + j1] > 0 && s.Cells[i1 * 9 + j1] < 10);
            }

        for (int r = 0; r < 9; r++)
        {
            var r1 = r;
            th = th.Where(t => Z3Methods.Distinct(indices.Select(j => t.Cells[r1 * 9 + j]).ToArray()));
        }
        for (int c = 0; c < 9; c++)
        {
            var c1 = c;
            th = th.Where(t => Z3Methods.Distinct(indices.Select(i => t.Cells[i * 9 + c1]).ToArray()));
        }
        for (int b = 0; b < 9; b++)
        {
            var b1 = b;
            var idx = (b1 / 3) * 3 * 9 + (b1 % 3) * 3;
            th = th.Where(t => Z3Methods.Distinct(new int[] {
                t.Cells[idx],      t.Cells[idx + 1],  t.Cells[idx + 2],
                t.Cells[idx + 9],  t.Cells[idx + 10], t.Cells[idx + 11],
                t.Cells[idx + 18], t.Cells[idx + 19], t.Cells[idx + 20] }));
        }

        for (int k = 0; k < 81; k++)
        {
            var k1 = k;
            var v = int.Parse(puzzle.Substring(k1, 1));
            if (v != 0) th = th.Where(t => t.Cells[k1] == v);
        }
        return th;
    }

    // Le meme encodage, sur le type d'environnement porte par l'attribut.
    public static Theorem<SudokuArrayAvecRegle> ChargerAvecRegle(Z3Context ctx, string puzzle)
    {
        var th = ctx.NewTheorem<SudokuArrayAvecRegle>();
        var indices = Enumerable.Range(0, 9).ToArray();

        for (int i = 0; i < 9; i++)
            for (int j = 0; j < 9; j++)
            {
                var i1 = i; var j1 = j;
                th = th.Where(s => s.Cells[i1 * 9 + j1] > 0 && s.Cells[i1 * 9 + j1] < 10);
            }

        for (int r = 0; r < 9; r++)
        {
            var r1 = r;
            th = th.Where(t => Z3Methods.Distinct(indices.Select(j => t.Cells[r1 * 9 + j]).ToArray()));
        }
        for (int c = 0; c < 9; c++)
        {
            var c1 = c;
            th = th.Where(t => Z3Methods.Distinct(indices.Select(i => t.Cells[i * 9 + c1]).ToArray()));
        }
        for (int b = 0; b < 9; b++)
        {
            var b1 = b;
            var idx = (b1 / 3) * 3 * 9 + (b1 % 3) * 3;
            th = th.Where(t => Z3Methods.Distinct(new int[] {
                t.Cells[idx],      t.Cells[idx + 1],  t.Cells[idx + 2],
                t.Cells[idx + 9],  t.Cells[idx + 10], t.Cells[idx + 11],
                t.Cells[idx + 18], t.Cells[idx + 19], t.Cells[idx + 20] }));
        }

        for (int k = 0; k < 81; k++)
        {
            var k1 = k;
            var v = int.Parse(puzzle.Substring(k1, 1));
            if (v != 0) th = th.Where(t => t.Cells[k1] == v);
        }
        return th;
    }

    public static string Grille(List<int> cells) =>
        string.Join(" | ", Enumerable.Range(0, 9).Select(r =>
            string.Join("", Enumerable.Range(0, 9).Select(c => cells[r * 9 + c].ToString()))));
}


warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

### L'encodage, et pourquoi il est ecrit deux fois

`Sudoku09.Charger` et `Sudoku09.ChargerAvecRegle` font presque la meme chose : la meme grille, les memes contraintes de domaine, les memes contraintes de structure, les memes indices. La seule difference est le type d'environnement retourne, qui porte - ou non - l'attribut de reecriture.

La duplication n'est pas une maladresse : le binding resout les membres de l'environnement sur le **type concret** de l'arbre d'expressions. Un encodeur generique echoue sur « Unknown parameter encountered: Cells. » (constate par une sonde hors carnet) - la meme contrainte que pour la propriete heritee. Un point d'extension se paie donc aussi en contraintes de conception.

In [6]:
Console.WriteLine("=== Reference : le meme enonce, sans reecrivain global");
using (var ctx = new Z3Context())
{
    var solution = Sudoku09.Charger(ctx, Puzzle).Solve();
    Console.WriteLine($"[DSL ] {Sudoku09.Grille(solution.Cells)}");
}

Console.WriteLine();
Console.WriteLine("=== Le meme enonce, avec le reecrivain global (interdit 5 en case 0)");
using (var ctx = new Z3Context())
{
    var solution = Sudoku09.ChargerAvecRegle(ctx, Puzzle).Solve();
    Console.WriteLine($"[DSL ] Solve -> {(solution == null ? "null (aucune solution)" : Sudoku09.Grille(solution.Cells))}");
}

Console.WriteLine();
Console.WriteLine("=== Le coeur UNSAT : quelles contraintes sont en conflit ?");
using (var ctx = new Z3Context())
{
    var explication = Sudoku09.ChargerAvecRegle(ctx, Puzzle).Explain();
    Console.WriteLine($"Statut : {explication.Status}");
    foreach (var c in explication.UnsatCore) Console.WriteLine($"  core : {c}");
}

Console.WriteLine();
Console.WriteLine($"Cases 0 et 71 : valeurs dans la grille = {Puzzle.Substring(0, 1)} et {Puzzle.Substring(71, 1)}");

=== Reference : le meme enonce, sans reecrivain global


[DSL ] 534678912 | 672195348 | 198342567 | 859761423 | 426853791 | 713924856 | 961537284 | 287419635 | 345286179


=== Le meme enonce, avec le reecrivain global (interdit 5 en case 0)


  [ForbidValueRewriter] 138 contraintes recues ; ajout : (s.Cells.get_Item(0) != 5)


[DSL ] Solve -> null (aucune solution)


...

  [ForbidValueRewriter] 138 contraintes recues ; ajout : (s.Cells.get_Item(0) != 5)


Statut : Unsatisfiable


  core : #108: (t.Cells.get_Item(value(Submission#6+Sudoku09+<>c__DisplayClass1_5).k1) == value(Submission#6+Sudoku09+<>c__DisplayClass1_5).v)


  core : #138: (s.Cells.get_Item(0) != 5)


Cases 0 et 71 : valeurs dans la grille = 5 et 5


### Lecture de la section 1

Le meme enonce, resolu deux fois, donne **deux resultats differents** : une grille complete sans le reecrivain, `null` avec. La regle a change le verdict, et elle l'a change par un chemin **visible** : le reecrivain affiche la contrainte qu'il a construite, et le coeur UNSAT la nomme.

Le coeur UNSAT est la preuve la plus forte, parce qu'elle est **falsifiable** et **numerable** :

- le reecrivain a recu **138 contraintes** : 81 contraintes de domaine, 27 contraintes de structure (9 par ligne, 9 par colonne, 9 par boite), et les 30 indices de la grille ;
- la contrainte qu'il a ajoutee est la **#138** : elle porte l'index 138 parce qu'elle vient apres les 138 precedentes ;
- le coeur nomme la **#108** et la **#138** : la #108 est le premier indice de la grille (108 = 81 + 27), qui fixe la case 0 a 5 ; la #138 est la regle qui interdit 5 dans cette case. Les deux lignes de sortie « Cases 0 et 71 » rappellent que la case 0 vaut bien 5.

Ce que ce carnet ne fait pas, c'est **inventer** une contradiction : l'index 0 de la grille vaut 5, la regle interdit 5, le solveur ne peut pas satisfaire les deux. Un reecrivain global n'est donc pas une surcouche decorative : c'est une **contrainte reelle**, qui compte dans le coeur UNSAT comme les contraintes ecrites a la main.

La contrepartie, elle, est mesuree dans la sortie : l'encodage est **duplique** dans `Sudoku09.Charger` et `Sudoku09.ChargerAvecRegle`. Ce n'est pas une maladresse du carnet, c'est une propriete du binding : il resout les membres de l'environnement sur le **type concret** de l'arbre d'expressions, si bien qu'un encodeur generique - ou une propriete heritee d'une classe de base - echoue sur « Unknown parameter encountered: Cells. » - message constate par une sonde hors carnet, pas lu dans une sortie de ce carnet. Un point d'extension ne se paie pas seulement en lignes de code : il se paie en **contraintes de conception**.

### Exercice 1 - faire varier la regle et lire le verdict

La regle de la section 1 vise la case 0 et interdit la valeur 5. Vous allez la faire varier : c'est le rapport entre la valeur interdite et la solution qui decide si le theoreme reste soluble.

Objectif : produire, dans une seule ligne, le verdict du solveur pour une regle de votre choix, et dire **quand** le verdict change. Deux cas a distinguer :

- une regle qui interdit, sur la case visee, **la valeur que la solution lui donne** contredit cette solution : le solveur ne rend plus de solution ;
- une regle qui interdit **une autre valeur** ne contredit rien : le solveur rend toujours la meme solution.

Le statut de la case visee - indice ou libre - n'est pas le bon critere : le verdict ne depend que du rapport entre la valeur interdite et la valeur que la solution donne a la case.

*Indice* : la regle se regle par les deux champs de `Regle` ; l'encodeur `Sudoku09.ChargerAvecRegle` fait le reste. La valeur que la solution donne a chaque case se lit sur la grille de reference de la section 1 ; les valeurs imposees se reperent aussi dans la chaine `Puzzle` (les chiffres non nuls).

In [7]:
// --- Exercice 1 ---
// Faites varier la regle (case visee, valeur interdite) et renvoyez le verdict
// du solveur en une ligne.
static string VerdictAvecRegle()
{
    // TODO etudiant : regler Regle.CaseInterdite / Regle.ValeurInterdite, resoudre
    // avec Sudoku09.ChargerAvecRegle, et renvoyer le verdict ("null" ou la grille)
    Console.WriteLine("Exercice a completer : regler la regle et renvoyer le verdict");
    return "a completer";
}

// La regle de la section 1 : case 0, valeur 5 interdite.
Regle.CaseInterdite = 0;
Regle.ValeurInterdite = 5;
Console.WriteLine($"Exercice 1 : verdict = {VerdictAvecRegle()}");

Exercice a completer : regler la regle et renvoyer le verdict


Exercice 1 : verdict = a completer


## 2. Le reecrivain de predicat : traduire une methode que le DSL ne connait pas

Le deuxieme point d'extension ne travaille pas sur le theoreme entier mais sur **un appel de methode**. Le visiteur d'expressions du binding ne connait qu'un ensemble ferme de predicats (les "magic methods" du DSL, comme `Distinct` ou `Sum`). Appeler une methode ordinaire dans une contrainte `.Where(...)` echoue donc :

```
NotSupportedException : Unknown method call:Boolean SansRegle(Int32, Int32)
```

L'attribut `[TheoremPredicateRewriter(typeof(R))]` se pose sur **la methode** et dit au binding comment remplacer cet appel par un appel que lui sait traduire. Le reecrivain recoit l'appel (`MethodCallExpression`) et renvoie l'appel de remplacement.

Nous allons ecrire un predicat maison - « ces deux cases sont differentes » - et le traduire par l'operateur `Distinct` du DSL, qui exprime exactement la meme chose. Pour que la traduction soit **falsifiable**, on l'applique a deux cases dont les valeurs sont fixees par la grille : si la traduction est correcte, le theoreme devient insoluble, et le coeur UNSAT nomme la contrainte traduite.

In [8]:
using System.Linq.Expressions;

// --- Les predicats ---
// Une methode ordinaire : le visiteur d'expressions ne sait pas la traduire.
public static class Predicats
{
    public static bool SansRegle(int a, int b) => a != b;

    // La meme semantique, mais declaree traduisible : l'attribut dit au binding
    // par quoi la remplacer. Comme les predicats du DSL, la methode n'est
    // utilisable que dans une expression de theorie.
    [TheoremPredicateRewriter(typeof(ToDistinctRewriter))]
    public static bool PasLesMemes(int a, int b) => throw new NotSupportedException("Predicate for use in query expressions only.");
}

// --- Le reecrivain de predicat ---
// Il recoit l'appel et renvoie l'appel a traduire a sa place : ici, `Distinct`
// sur les deux arguments, que le DSL sait traduire.
public class ToDistinctRewriter : ITheoremPredicateRewriter
{
    public MethodCallExpression Rewrite(MethodCallExpression call)
    {
        var definition = typeof(Z3Methods).GetMethod("Distinct")!;
        var ferme = definition.MakeGenericMethod(typeof(int));
        return Expression.Call(ferme, Expression.NewArrayInit(typeof(int), call.Arguments));
    }
}


warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



In [9]:
Console.WriteLine("=== Un predicat inconnu du DSL, SANS reecrivain");
try
{
    using var ctx = new Z3Context();
    var solution = Sudoku09.Charger(ctx, Puzzle)
        .Where(t => Predicats.SansRegle(t.Cells[0], t.Cells[1]))
        .Solve();
    Console.WriteLine($"[DSL ] {Sudoku09.Grille(solution.Cells)}");
}
catch (NotSupportedException ex)
{
    Console.WriteLine($"[DSL ] NotSupportedException : {ex.Message}");
}

Console.WriteLine();
Console.WriteLine("=== Le predicat traduit, applique a deux cases fixees par la grille");
Console.WriteLine($"Cases 0 et 71 : valeurs dans la grille = {Puzzle.Substring(0, 1)} et {Puzzle.Substring(71, 1)}");
using (var ctx = new Z3Context())
{
    var solution = Sudoku09.Charger(ctx, Puzzle)
        .Where(t => Predicats.PasLesMemes(t.Cells[0], t.Cells[71]))
        .Solve();
    Console.WriteLine($"[DSL ] Solve -> {(solution == null ? "null (aucune solution)" : Sudoku09.Grille(solution.Cells))}");
}

Console.WriteLine();
Console.WriteLine("=== Le coeur UNSAT du predicat traduit");
using (var ctx = new Z3Context())
{
    var explication = Sudoku09.Charger(ctx, Puzzle)
        .Where(t => Predicats.PasLesMemes(t.Cells[0], t.Cells[71]))
        .Explain();
    Console.WriteLine($"Statut : {explication.Status}");
    foreach (var c in explication.UnsatCore) Console.WriteLine($"  core : {c}");
}

Console.WriteLine();
Console.WriteLine("=== Le meme predicat traduit, sur deux cases de valeurs differentes (0 et 1)");
using (var ctx = new Z3Context())
{
    var solution = Sudoku09.Charger(ctx, Puzzle)
        .Where(t => Predicats.PasLesMemes(t.Cells[0], t.Cells[1]))
        .Solve();
    Console.WriteLine($"[DSL ] {(solution == null ? "null (aucune solution)" : Sudoku09.Grille(solution.Cells))}");
}

=== Un predicat inconnu du DSL, SANS reecrivain


[DSL ] NotSupportedException : Unknown method call:Boolean SansRegle(Int32, Int32)


=== Le predicat traduit, applique a deux cases fixees par la grille


Cases 0 et 71 : valeurs dans la grille = 5 et 5


[DSL ] Solve -> null (aucune solution)


...

  core : #134: (t.Cells.get_Item(value(Submission#6+Sudoku09+<>c__DisplayClass0_5).k1) == value(Submission#6+Sudoku09+<>c__DisplayClass0_5).v)


  core : #138: PasLesMemes(t.Cells.get_Item(0), t.Cells.get_Item(71))


=== Le meme predicat traduit, sur deux cases de valeurs differentes (0 et 1)


[DSL ] 534678912 | 672195348 | 198342567 | 859761423 | 426853791 | 713924856 | 961537284 | 287419635 | 345286179



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

### Lecture de la section 2

Les quatre mesures se lisent ensemble.

**Le predicat non traduit echoue au moment de la traduction en Z3, pas a la resolution.** Le message `Unknown method call:Boolean SansRegle(Int32, Int32)` dit que le visiteur d'expressions a refuse une methode qu'il ne connait pas. Le carnet n'a pas « echoue » : il a **montre** la frontiere du DSL.

**Le predicat traduit est une vraie contrainte.** Applique aux cases 0 et 71 - deux cases dont la grille fixe la valeur a 5 - il rend le theoreme insoluble, et le coeur UNSAT nomme les trois contraintes en conflit : les deux indices qui fixent ces cases (#108 et #134) et la contrainte traduite (#138, `PasLesMemes(t.Cells.get_Item(0), t.Cells.get_Item(71))`).

Notez la forme du nom dans le coeur : c'est l'appel **d'origine**, pas sa traduction. Le coeur UNSAT designe les contraintes du theoreme, pas les formules Z3 ; la traduction a eu lieu en aval, au moment de la visite.

**La traduction discriminate.** Le meme predicat, applique aux cases 0 et 1 (valeurs 5 et 3 dans la grille), laisse le theoreme soluble et rend la meme grille que la reference. Un predicat traduit n'est donc pas une contrainte vide : elle accepte un couple et en refuse un autre, exactement comme le fait `Distinct`.

Ce que ce mecanisme change, enfin, c'est la **frontiere du DSL** : on peut maintenant ecrire une contrainte dans le langage du domaine, avec un predicat nomme, et la faire compiler par le binding. Le prix a payer est explicite : le predicat doit etre traduit par un reecrivain ecrit a la main, et ce reecrivain est du code que vous maintenez.

### Exercice 2 - ecrire votre propre predicat et son reecrivain

Vous avez vu un predicat traduit vers `Distinct`. Ecrivez le votre : un predicat de votre choix, son reecrivain, et l'application du predicat sur deux cases de la grille.

Objectif : produire une ligne qui dit ce que le solveur repond quand votre predicat est applique aux deux cases que vous avez choisies - en verifiant d'abord qu'il repond quelque chose, c'est-a-dire que votre predicat a bien ete traduit.

*Indice* : un predicat du DSL ne s'utilise que dans une expression de theorie (il leve une exception s'il est appele directement) ; son reecrivain renvoie un `MethodCallExpression` ; l'attribut se pose sur la methode. Declarez vos types dans une cellule dediee, avant la cellule qui les utilise.

In [10]:
// --- Exercice 2 ---
// Ecrivez votre propre predicat ET son reecrivain, puis appliquez-le a deux
// cases de votre choix. Renvoyez ce que repond le solveur.
static string MonPredicatTraduit(int caseA, int caseB)
{
    // TODO etudiant : declarer le predicat et son reecrivain dans une nouvelle
    // cellule, les appliquer a deux cases, et renvoyer le verdict du solveur
    Console.WriteLine("Exercice a completer : ecrire le predicat et son reecrivain");
    return "a completer";
}

Console.WriteLine($"Exercice 2 : verdict = {MonPredicatTraduit(0, 1)}");

Exercice a completer : ecrire le predicat et son reecrivain


Exercice 2 : verdict = a completer


## 3. Les gardes-fous du contrat

Un point d'extension donne du pouvoir : celui d'ajouter des contraintes, ou de remplacer un predicat. Le binding ne laisse pas ce pouvoir sans controle. Trois garde-fous sont mesures ci-dessous, et reproduits **verbatim** :

1. un attribut qui pointe sur un type qui n'implemente pas l'interface attendue est **refuse** ;
2. un reecrivain qui renvoie son entree sans rien reecrire est **refuse**, pour eviter une boucle infinie dans la visite des expressions ;
3. la meme verification existe pour le reecrivain global.

Ces messages ne sont pas des accidents : ce sont les **conditions du contrat** des deux interfaces, ecrites par le binding.

In [11]:
using System.Linq.Expressions;

// --- Les reecrivains fautifs, pour declencher les gardes ---
// Un type qui n'implemente pas l'interface du reecrivain de predicat...
public class NotARewriter { }

// ...et un reecrivain qui renvoie son entree sans rien changer.
public class NoOpRewriter : ITheoremPredicateRewriter
{
    public MethodCallExpression Rewrite(MethodCallExpression call) => call;
}

// Les predicats fautifs qui pointent sur ces deux types.
public static class PredicatsFautifs
{
    [TheoremPredicateRewriter(typeof(NotARewriter))]
    public static bool CasInterface(int a, int b) => throw new NotSupportedException("Predicate for use in query expressions only.");

    [TheoremPredicateRewriter(typeof(NoOpRewriter))]
    public static bool CasNoOp(int a, int b) => throw new NotSupportedException("Predicate for use in query expressions only.");
}


warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime



In [12]:
Console.WriteLine("=== Garde 1 : un type de reecrivain qui n'implemente pas l'interface");
try
{
    using var ctx = new Z3Context();
    Sudoku09.Charger(ctx, Puzzle)
        .Where(t => PredicatsFautifs.CasInterface(t.Cells[0], t.Cells[1]))
        .Solve();
    Console.WriteLine("[DSL ] aucune exception (inattendu)");
}
catch (InvalidOperationException ex)
{
    Console.WriteLine($"[DSL ] InvalidOperationException : {ex.Message}");
}

Console.WriteLine();
Console.WriteLine("=== Garde 2 : un reecrivain qui renvoie son entree sans rien reecrire");
try
{
    using var ctx = new Z3Context();
    Sudoku09.Charger(ctx, Puzzle)
        .Where(t => PredicatsFautifs.CasNoOp(t.Cells[0], t.Cells[1]))
        .Solve();
    Console.WriteLine("[DSL ] aucune exception (inattendu)");
}
catch (InvalidOperationException ex)
{
    Console.WriteLine($"[DSL ] InvalidOperationException : {ex.Message}");
}

Console.WriteLine();
Console.WriteLine("=== Garde 3 : un type de reecrivain global qui n'implemente pas l'interface");
try
{
    using var ctx = new Z3Context();
    ctx.NewTheorem<SudokuArrayGarde>().Where(t => t.Cells[0] == 5).Solve();
    Console.WriteLine("[DSL ] aucune exception (inattendu)");
}
catch (InvalidOperationException ex)
{
    Console.WriteLine($"[DSL ] InvalidOperationException : {ex.Message}");
}

=== Garde 1 : un type de reecrivain qui n'implemente pas l'interface


[DSL ] InvalidOperationException : Invalid predicate rewriter type definition. Did you implement ITheoremPredicateRewriter?


=== Garde 2 : un reecrivain qui renvoie son entree sans rien reecrire


[DSL ] InvalidOperationException : The expression tree rewriter of type NoOpRewriter did not perform any rewrite. Aborting compilation to avoid infinite looping.


=== Garde 3 : un type de reecrivain global qui n'implemente pas l'interface


[DSL ] InvalidOperationException : Invalid global rewriter type definition. Did you implement ITheoremGlobalRewriter?



warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' de 'System.Linq.Expressions', il se peut que vous deviez fournir une stratégie runtime

warning CS1701: En supposant que la référence d'assembly 'System.Linq.Expressions, Version=8.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' utilisée par 'Z3.Linq' correspond à l'identité 'System.Linq.Expressions, Version=10.0.0.0, 

### Lecture de la section 3

Les trois messages sont **le contrat du binding**, ecrit dans ses mots.

- La garde 1 et la garde 3 disent la meme chose des deux points d'extension : l'attribut doit designer un type qui implemente l'interface attendue. Le binding verifie le type **avant** de l'instancier - il refuse donc un reecrivain mal forme sans jamais lancer de code de votre part.
- La garde 2 est d'une autre nature : elle ne verifie pas une forme, elle verifie un **effet**. Un reecrivain qui renvoie son appel sans le modifier ferait boucler la visite de l'expression ; plutot que de laisser tourner, le binding arrete et le dit. Le message nomme le type fautif, ce qui permet de le trouver sans chercher.

Ces trois garde-fous sont **l'argument** qui rend le point d'extension utilisable en production : le pouvoir d'ecrire ses propres regles est un pouvoir **encadre**. Vous pouvez ecrire une regle qui se trompe ; vous ne pouvez pas ecrire une regle qui fasse boucler la visite des expressions : le binding l'arrete avant, et le dit.

### Exercice 3 - declencher une garde et la rapporter

Les trois garde-fous de la section 3 sont reproductibles a la demande. Reproduisez-en une, et rapportez le message exact.

Objectif : ecrire une fonction qui declenche volontairement la garde demandee, qui attrape l'`InvalidOperationException`, et qui renvoie son message - sans que l'exception se propage.

*Indice* : la garde 1 se declenche avec `PredicatsFautifs.CasInterface`, la garde 2 avec `PredicatsFautifs.CasNoOp`, la garde 3 avec le type d'environnement `SudokuArrayGarde`.

In [13]:
// --- Exercice 3 ---
// Declenchez volontairement UNE des trois gardes de la section 3, et renvoyez son
// message verbatim.
static string MessageDeGarde(int numeroDeGarde)
{
    // TODO etudiant : appeler le theoreme qui declenche la garde demandee,
    // attraper l'InvalidOperationException, et renvoyer son message
    Console.WriteLine("Exercice a completer : renvoyer le message de la garde");
    return "a completer";
}

Console.WriteLine($"Exercice 3 : message = « {MessageDeGarde(1)} »");

Exercice a completer : renvoyer le message de la garde


Exercice 3 : message = « a completer »


## Conclusion

Le binding `Z3.Linq` ne se contente pas de traduire un theoreme : il offre **deux points d'extension** pour ecrire les regles de traduction soi-meme.

Quatre choses a retenir :

1. **Le reecrivain global agit sur toutes les contraintes.** Il est declare sur le **type d'environnement**, recoit la liste complete des contraintes et renvoie la liste a faire assertor. Ici, il a recu 138 contraintes et en a ajoute une : le meme enonce est passe d'une grille complete a `null`, et le coeur UNSAT nomme la regle qu'il a ecrite (#138) en face de l'indice qu'elle contredit (#108).
2. **Le reecrivain de predicat agit sur un appel.** Il est declare sur la **methode**, et remplace un predicat que le DSL ne connait pas par un predicat qu'il sait traduire. Le predicat non traduit echoue au moment de la traduction en Z3 (« Unknown method call »), le predicat traduit produit une vraie contrainte - un coeur UNSAT la nomme, et un couple de cases de valeurs differentes passe.
3. **Les deux mechanismes sont encadres.** Type qui n'implemente pas l'interface, reecrivain qui ne reecrit rien : le binding refuse et le dit, avec un message qui nomme le coupable.
4. **Un point d'extension a un cout de conception.** Le binding resout les membres de l'environnement sur le type concret : l'encodage a ete duplique dans ce carnet pour cette raison, et la propriete de l'environnement doit etre declaree sur le type lui-meme.

Ces deux extensions ouvrent un pan du DSL qui n'etait exerce par aucun carnet : ecrire ses propres regles, en amont de la traduction, dans le langage du probleme.

**Navigation** : [Index SymbolicAI](../../README.md) | [Serie Z3](README.md)